# Paper Figures: Training Curves and Results

This notebook generates all figures for the paper:
**"Mitigating Position-Shift Failures in Text-Based Modular Arithmetic via Position Curriculum and Template Diversity"**

## Instructions

### Option 1: Google Colab (Recommended)
1. Upload this notebook to Google Colab
2. Upload `training_curves.json` and `final_paper_results.json` to Colab Files panel
3. Run all cells (Runtime → Run all)
4. Download generated figures from Files panel

### Option 2: Local Environment
1. Install dependencies: `pip install matplotlib seaborn numpy`
2. Place `training_curves.json` and `final_paper_results.json` in same directory
3. Run all cells
4. Figures saved to current directory

## Requirements
- `training_curves.json` (443 KB) - all training curves
- `final_paper_results.json` (5.6 KB) - aggregated results

## Output
- `figure1_training_curves_all.png` - Training dynamics across all experiments
- `figure2_comparison_i1_002a_vs_baseline.png` - Direct comparison
- `figure3_position_breakdown.png` - Position-shift robustness breakdown
- `figure4_final_performance_summary.png` - Summary bar charts
- `paper_figures.zip` - All figures packaged (300 DPI)

## Setup: Install Dependencies and Load Data

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install matplotlib seaborn numpy -q

import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Set plotting style
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

print('✅ Libraries imported')
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# Load data files
print('Loading data files...')

try:
    with open('training_curves.json') as f:
        curves = json.load(f)
    print(f'✅ Loaded training_curves.json: {len(curves)} experiments')
except FileNotFoundError:
    print('❌ Error: training_curves.json not found')
    print('   Please upload training_curves.json to the same directory')
    raise

try:
    with open('final_paper_results.json') as f:
        results = json.load(f)
    print(f'✅ Loaded final_paper_results.json: {len(results)} experiments')
except FileNotFoundError:
    print('❌ Error: final_paper_results.json not found')
    print('   Please upload final_paper_results.json to the same directory')
    raise

# Define experiment metadata
COLORS = {
    'baseline_001': 'C0',
    'i1_001_1': 'C1',
    'i1_002_alibi': 'C2',
    'i1_002a': 'C3',
}

LABELS = {
    'baseline_001': 'Baseline',
    'i1_001_1': 'I1_001_1 (Position)',
    'i1_002_alibi': 'I1_002_ALIBI',
    'i1_002a': 'I1_002a (Main)',
}

EXP_ORDER = ['baseline_001', 'i1_001_1', 'i1_002_alibi', 'i1_002a']

print('✅ Setup complete')

## Figure 1: Training Curves (All Metrics)

Shows training dynamics across all experiments:
- Top-left: Training accuracy
- Top-right: Eval-A (in-distribution)
- Bottom-left: Eval-B (position shift)
- Bottom-right: Eval-C0 (template OOD)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Curves - All Experiments (3 seeds each)', fontsize=16, fontweight='bold')

# Training Accuracy
ax = axes[0, 0]
for exp_name in EXP_ORDER:
    if exp_name not in curves:
        continue
    exp_data = curves[exp_name]
    for i, (seed_name, seed_data) in enumerate(exp_data['seeds'].items()):
        steps = seed_data['train']['steps']
        accs = seed_data['train']['accuracy']
        lbl = LABELS[exp_name] if i == 0 else None
        ax.plot(steps, accs, color=COLORS[exp_name], alpha=0.6, linewidth=1.0, label=lbl)
ax.set_xlabel('Training Steps')
ax.set_ylabel('Training Accuracy')
ax.set_title('Train Accuracy vs Steps')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Eval-A
ax = axes[0, 1]
for exp_name in EXP_ORDER:
    if exp_name not in curves:
        continue
    exp_data = curves[exp_name]
    for i, (seed_name, seed_data) in enumerate(exp_data['seeds'].items()):
        eval_data = seed_data['eval_A']
        steps = [d['step'] for d in eval_data if d['accuracy'] is not None]
        accs = [d['accuracy'] for d in eval_data if d['accuracy'] is not None]
        lbl = LABELS[exp_name] if i == 0 else None
        ax.plot(steps, accs, 'o-', color=COLORS[exp_name], alpha=0.6, markersize=3, linewidth=1.0, label=lbl)
ax.set_xlabel('Training Steps')
ax.set_ylabel('Eval-A Accuracy')
ax.set_title('Eval-A (In-Distribution) vs Steps')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Eval-B
ax = axes[1, 0]
for exp_name in EXP_ORDER:
    if exp_name not in curves:
        continue
    exp_data = curves[exp_name]
    for i, (seed_name, seed_data) in enumerate(exp_data['seeds'].items()):
        eval_data = seed_data['eval_B']
        steps = [d['step'] for d in eval_data if d['accuracy'] is not None]
        accs = [d['accuracy'] for d in eval_data if d['accuracy'] is not None]
        lbl = LABELS[exp_name] if i == 0 else None
        ax.plot(steps, accs, 'o-', color=COLORS[exp_name], alpha=0.6, markersize=3, linewidth=1.0, label=lbl)
ax.set_xlabel('Training Steps')
ax.set_ylabel('Eval-B Accuracy')
ax.set_title('Eval-B (Position Shift) vs Steps')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Eval-C0
ax = axes[1, 1]
for exp_name in EXP_ORDER:
    if exp_name not in curves:
        continue
    exp_data = curves[exp_name]
    for i, (seed_name, seed_data) in enumerate(exp_data['seeds'].items()):
        eval_data = seed_data['eval_C0']
        steps = [d['step'] for d in eval_data if d['accuracy'] is not None]
        accs = [d['accuracy'] for d in eval_data if d['accuracy'] is not None]
        lbl = LABELS[exp_name] if i == 0 else None
        ax.plot(steps, accs, 'o-', color=COLORS[exp_name], alpha=0.6, markersize=3, linewidth=1.0, label=lbl)
ax.set_xlabel('Training Steps')
ax.set_ylabel('Eval-C0 Accuracy')
ax.set_title('Eval-C0 (Template OOD) vs Steps')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('figure1_training_curves_all.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Figure 1 saved: figure1_training_curves_all.png')

## Figure 2: I1_002a vs Baseline Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('I1_002a vs Baseline Comparison', fontsize=16, fontweight='bold')

comparison_exps = ['baseline_001', 'i1_002a']
metrics = [
    ('eval_A', 'Eval-A Accuracy', 'In-Distribution'),
    ('eval_B', 'Eval-B Accuracy', 'Position Shift'),
    ('eval_C0', 'Eval-C0 Accuracy', 'Template OOD'),
]

for i, (metric_key, ylabel, title) in enumerate(metrics):
    ax = axes[i]
    for exp_name in comparison_exps:
        if exp_name not in curves:
            continue
        exp_data = curves[exp_name]
        for j, (seed_name, seed_data) in enumerate(exp_data['seeds'].items()):
            eval_data = seed_data[metric_key]
            steps = [d['step'] for d in eval_data if d['accuracy'] is not None]
            accs = [d['accuracy'] for d in eval_data if d['accuracy'] is not None]
            lbl = LABELS[exp_name] if j == 0 else None
            ax.plot(steps, accs, 'o-', color=COLORS[exp_name], alpha=0.6, markersize=4, linewidth=1.5, label=lbl)
    ax.set_xlabel('Training Steps')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('figure2_comparison_i1_002a_vs_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Figure 2 saved: figure2_comparison_i1_002a_vs_baseline.png')

## Figure 3: Position Breakdown (Eval-B)

Shows catastrophic cliff in baseline vs position invariance in I1_002a

In [ ]:
# Extract position data
positions = [0, 8, 16, 24, 32, 48, 64]
position_data = {}

for exp_name in EXP_ORDER:
    if exp_name not in results:
        continue
    exp_results = results[exp_name]
    per_pos = exp_results['eval_B']['per_position']
    means = []
    stds = []
    for pos in positions:
        # Keys are strings like "0", "8", "16" (not "pos_0")
        # Values are arrays [mean, std]
        pos_key = str(pos)
        if pos_key in per_pos:
            pos_data = per_pos[pos_key]
            means.append(pos_data[0])  # First element is mean
            stds.append(pos_data[1])   # Second element is std
        else:
            means.append(0)
            stds.append(0)
    position_data[exp_name] = {'means': means, 'stds': stds}

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(positions))
width = 0.2

for i, exp_name in enumerate(EXP_ORDER):
    if exp_name not in position_data:
        continue
    data = position_data[exp_name]
    offset = (i - 1.5) * width
    ax.bar(x + offset, data['means'], width, yerr=data['stds'], 
           label=LABELS[exp_name], color=COLORS[exp_name], alpha=0.8, capsize=3)

ax.set_xlabel('Expression Position (characters from start)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Eval-B: Position Shift Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Pos {p}' for p in positions])
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

# Annotate cliff
baseline_data = position_data.get('baseline_001')
if baseline_data and len(baseline_data['means']) > 0:
    cliff_drop = baseline_data['means'][0] - baseline_data['means'][-1]
    ax.annotate(f'Catastrophic cliff:\n{cliff_drop*100:.0f}% drop', 
                xy=(0, baseline_data['means'][0]), xytext=(1.5, 0.7),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                fontsize=10, color='red',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('figure3_position_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Figure 3 saved: figure3_position_breakdown.png')

## Figure 4: Final Performance Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Final Performance Summary (Mean ± Std)', fontsize=16, fontweight='bold')

metrics = [
    ('eval_A', 'mean', 'Eval-A\n(In-Distribution)'),
    ('eval_B', 'mean', 'Eval-B\n(Position Shift)'),
    ('eval_C0', 'mean', 'Eval-C0\n(Template OOD)'),
]

for i, (metric_key, sub_key, title) in enumerate(metrics):
    ax = axes[i]
    means = []
    stds = []
    labels_list = []
    colors_list = []
    for exp_name in EXP_ORDER:
        if exp_name not in results:
            continue
        metric_data = results[exp_name][metric_key]
        means.append(metric_data[sub_key])
        stds.append(metric_data['std'])
        labels_list.append(LABELS[exp_name].replace(' ', '\n', 1))
        colors_list.append(COLORS[exp_name])
    
    x = np.arange(len(means))
    bars = ax.bar(x, means, yerr=stds, color=colors_list, alpha=0.8, capsize=5)
    
    # Add labels
    for j, (bar, mean, std) in enumerate(zip(bars, means, stds)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + std + 0.02,
                f'{mean*100:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_list, fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figure4_final_performance_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Figure 4 saved: figure4_final_performance_summary.png')

## Summary Statistics

In [ ]:
print('='*80)
print('FINAL RESULTS SUMMARY')
print('='*80)
print()

for exp_name in EXP_ORDER:
    if exp_name not in results:
        continue
    
    # Access data directly (flat structure)
    eval_a = results[exp_name]['eval_A']
    eval_b = results[exp_name]['eval_B']
    eval_c0 = results[exp_name]['eval_C0']
    eval_c1 = results[exp_name]['eval_C1']
    
    print(f'{LABELS[exp_name]}:')
    print(f'  Eval-A: {eval_a["mean"]*100:.1f} ± {eval_a["std"]*100:.1f}%')
    print(f'  Eval-B: {eval_b["mean"]*100:.1f} ± {eval_b["std"]*100:.1f}%')
    print(f'  Eval-C0: {eval_c0["mean"]*100:.1f} ± {eval_c0["std"]*100:.1f}%')
    if eval_c1:
        print(f'  Eval-C1: {eval_c1["mean"]*100:.1f} ± {eval_c1["std"]*100:.1f}%')
    print()

# Calculate improvements
baseline = results['baseline_001']
i1_002a = results['i1_002a']

print('='*80)
print('I1_002a IMPROVEMENTS vs BASELINE')
print('='*80)
print()

eval_b_improvement = (i1_002a['eval_B']['mean'] - baseline['eval_B']['mean']) * 100
eval_c0_improvement = (i1_002a['eval_C0']['mean'] - baseline['eval_C0']['mean']) * 100

print(f'Eval-B (Position Shift): +{eval_b_improvement:.1f} percentage points')
print(f'Eval-C0 (Template OOD): +{eval_c0_improvement:.1f} percentage points')
print()

print('✅ All figures generated successfully!')
print()
print('Generated files:')
print('  1. figure1_training_curves_all.png')
print('  2. figure2_comparison_i1_002a_vs_baseline.png')
print('  3. figure3_position_breakdown.png')
print('  4. figure4_final_performance_summary.png')

## Download Figures (Optional)

Create a zip file with all figures for easy download

In [ ]:
import zipfile

figure_files = [
    'figure1_training_curves_all.png',
    'figure2_comparison_i1_002a_vs_baseline.png',
    'figure3_position_breakdown.png',
    'figure4_final_performance_summary.png',
]

with zipfile.ZipFile('paper_figures.zip', 'w') as zipf:
    for fig_file in figure_files:
        try:
            zipf.write(fig_file)
            print(f'✅ Added: {fig_file}')
        except FileNotFoundError:
            print(f'⚠️  Missing: {fig_file}')

print('\n✅ Created: paper_figures.zip')
print('\nAll figures ready for download!')